In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Vector Store Creation
The objective of this notebook is to store document embeddings in a FAISS vector index and test semantic retrieval.

FAISS provides efficient similarity search across large collections of embeddings. Instead of searching documents using keywords, the system retrieves chunks based on semantic similarity.

The resulting vector store will form the retrieval component of the Retrieval-Augmented Generation (RAG) pipeline.

In [1]:
# Install libraries required for vector storage and retrieval

!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q langchain-text-splitters

In [5]:
# Import libraries required for document loading, chunking,
# embedding generation and vector search

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

In [7]:
# Load the General Academic Regulations document

pdf_path = "/kaggle/input/datasets/makhosazanemthethwa/cos802-university-documents/General_Academic_Regulations_2026.pdf"

reader = PdfReader(pdf_path)


# Extract text from all pages

full_text = ""

for page in reader.pages:

    page_text = page.extract_text()

    if page_text:
        full_text += page_text + "\n"


# Display document statistics

print(f"Characters: {len(full_text):,}")
print(f"Words: {len(full_text.split()):,}")


# Recreate the chunking strategy validated in Notebook 02

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(full_text)

print(f"Chunks Created: {len(chunks)}")


# Load the Sentence Transformer model used in Notebook 03

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


# Generate embeddings for all chunks

embeddings = model.encode(
    chunks,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")


# Display embedding statistics

print(f"Embeddings Generated: {len(embeddings)}")
print(f"Embedding Dimension: {embeddings.shape[1]}")

Characters: 361,882
Words: 53,391
Chunks Created: 455


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings Generated: 455
Embedding Dimension: 384


In [8]:
# Create a FAISS vector index using L2 distance

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)


# Store all embedding vectors in the index

index.add(embeddings)


# Display the number of stored vectors

print(f"Vectors Stored: {index.ntotal}")

Vectors Stored: 455


In [11]:
# Define a sample question to test retrieval

query = "What is an honours degree?"

In [12]:
# Convert the query into the same vector space as the document chunks

query_embedding = model.encode([query])

query_embedding = np.array(query_embedding).astype("float32")

In [13]:
# Retrieve the five most semantically similar chunks

distances, indices = index.search(
    query_embedding,
    k=5
)


# Display retrieved chunk indices

print(indices)

[[152 132 121 119 131]]


In [14]:
# Display the retrieved document chunks

for rank, idx in enumerate(indices[0], start=1):

    print(f"\n===== Result {rank} =====\n")

    print(chunks[idx])


===== Result 1 =====

distinction 
 
1. A qualification (honours degree, postgraduate diploma or postgraduate certificate) will be 
awarded to a student only if the student has complied with all the requirements for the 
particular qualification, and has reached the level of competence prescribed for each module 
that is required for the qualification. 
 
2. No one is entitled to any privileges pertaining to an honours degree or postgraduate diploma 
before the qualification has been awarded to them at a graduation ceremony. In respect of 
professional registrations, faculties may issue confirmation letters to the relevant professional 
bodies prior to the graduation ceremonies.  
 
3. Qualification with distinction 
 
 Except where faculty regulations stipulate additional requirements, an honours degree or 
postgraduate diploma is awarded with distinction if a student meets the following 
requirements: 
 
3.1 obtained a weighted average of at least 75% (not rounded) , based on the mi

In [16]:
# Define a query about postgraduate study

query = "What is the minimum entry requirement for admission to a master's degree?"

# Convert the query into an embedding

query_embedding = model.encode([query])

query_embedding = np.array(query_embedding).astype("float32")


# Retrieve the five most similar chunks

distances, indices = index.search(
    query_embedding,
    k=5
)

for rank, idx in enumerate(indices[0], start=1):

    print(f"\n===== Result {rank} =====\n")

    print(chunks[idx])



===== Result 1 =====

33 
G31 Admission 
 
1. Compliance with the minimum requirements will not guarantee admission to a master’s 
degree at the University. In addition to institutional requirements, programme- specific 
admission requirements are determined by the relevant faculty board, approved by Senate 
and included in the relevant faculty yearbook. Furthermore, specific admission criteria may 
be applied, as the University only has a limited number of places available. A candidate may 
also be refused admission to a master’s degree, even if they meet the standard of 
competence in the discipline, if the relevant department lacks adequate supervision capacity. 
 
2. A candidate is admitted to a master’s programme only if they  are in possession of an 
appropriate honours degree, or an equivalent qualification (such as a professional bachelor’s 
degree or a postgraduate diploma) at NQF Level 8 in the same or a relevant field of study.

===== Result 2 =====

comprises at least 45 c

In [18]:
# Define a query relating to supplementary examinations

query = "What is a supplementary examination?"

query_embedding = model.encode([query])

query_embedding = np.array(query_embedding).astype("float32")

distances, indices = index.search(
    query_embedding,
    k=5
)

for rank, idx in enumerate(indices[0], start=1):

    print(f"\n===== Result {rank} =====\n")

    print(chunks[idx])



===== Result 1 =====

examination is 50%, subject to faculty regulations. 
 
3.5 Additional supplementary examinations are not available for students who are unable 
to write the examination at the times scheduled. This also applies to students who are 
permitted to write a special examination or test in the supplementary timeslot for 
religious reasons.  
 
3.6 In instances where students are unable to write the examination and supplementary 
examination as a consequence of a serious medical condition or an accident, such a 
student must apply for a special dispensation, with the support of t he dean, to the 
Registrar, who will make a final decision. 
 
3.7 Supplementary examinations cover the same subject matter as was the case for the 
examinations. 
 
4. Special examinations 
 
4.1 A student who is prevented from writing the standard examination due to illness or 
other qualifying circumstances, may be granted permission by the dean to write a 
special examination in the particul

# Observations

A FAISS vector index was successfully created using embeddings generated from the General Academic Regulations and Student Rules 2026 document.

The document was previously divided into 455 overlapping chunks, and each chunk was represented as a 384-dimensional embedding vector using the all-MiniLM-L6-v2 Sentence Transformer model. These embeddings were successfully stored within the FAISS vector database.

Several retrieval tests were conducted using natural language queries related to honours degrees, master's degree admission requirements and supplementary examinations. The retrieval results consistently returned semantically relevant chunks from the document collection.

For the query "What is the minimum entry requirement for admission to a master's degree?", the top-ranked results contained information specifying that admission to a master's programme requires an appropriate honours degree or an equivalent qualification. Similarly, queries relating to supplementary examinations returned chunks describing eligibility criteria, examination rules and special examination provisions.

The retrieval results demonstrate that the semantic search process successfully identified contextually relevant information without relying solely on exact keyword matching. Although some retrieved chunks did not provide direct definitions as the top-ranked result, the returned content remained highly relevant to the intent of the query.

These findings indicate that the retrieval component of the proposed Retrieval-Augmented Generation (RAG) architecture is functioning correctly and is capable of identifying appropriate source material from large university documents.

# Conclusion
Vector store creation and semantic retrieval testing were successfully completed.

Document embeddings generated from the General Academic Regulations and Student Rules 2026 document were stored in a FAISS vector index, enabling efficient similarity-based retrieval.

Retrieval experiments demonstrated that semantically relevant chunks could be identified using natural language questions relating to honours degrees, master's degree admission requirements and supplementary examinations. The retrieved content was closely aligned with the intent of the queries, confirming that the vector search process is operating effectively.

The successful creation of the vector store establishes the retrieval component of the RAG pipeline and provides the foundation for Retrieval-Augmented Generation.

The next stage of the project will focus on integrating a Large Language Model (LLM) with the retrieval component to generate answers using the retrieved document context.